In [1]:
import gymnasium
import gymnasium_robotics

gymnasium.register_envs(gymnasium_robotics)

import torch

from RLAlg.buffer import ReplayBuffer
from RLAlg.buffer.her import HindsightExperienceReplay

In [2]:
def setup_env(env_name):
    env = gymnasium.make(env_name)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)

    return env

env_name = "FetchReach-v4"

envs = gymnasium.vector.SyncVectorEnv([lambda: setup_env(env_name) for _ in range(20)])

In [3]:
obs_dim = envs.single_observation_space["observation"].shape
goal_dim = envs.single_observation_space["desired_goal"].shape

replay_buffer = ReplayBuffer(20, 10000)

In [4]:
replay_buffer.create_storage_space("observations", obs_dim, torch.float32)
replay_buffer.create_storage_space("achieved_goal", goal_dim, torch.float32)
replay_buffer.create_storage_space("next_observations", obs_dim, torch.float32)
replay_buffer.create_storage_space("next_achieved_goal", goal_dim, torch.float32)
replay_buffer.create_storage_space("desired_goal", goal_dim, torch.float32)
replay_buffer.create_storage_space("rewards", (), torch.float32)
replay_buffer.create_storage_space("dones", (), torch.float32)

key_batch = ["observations", "achieved_goal", "next_observations", "next_achieved_goal", "desired_goal", "rewards", "dones"]

In [8]:
obs_dict, info = envs.reset()
for _ in range(100):
    actions = envs.action_space.sample()
    
    next_obs_dict, rewards, terminations, truncations, infos = envs.step(actions)
    obs = obs_dict["observation"]
    achieved_goal = obs_dict["achieved_goal"]
    next_obs = next_obs_dict["observation"]
    next_achieved_goal = next_obs_dict["achieved_goal"]
    dones = terminations | truncations
    print(actions)
    print(obs[:, :3])
    print(achieved_goal)
    record = {
                "observations": obs,
                "next_observations": next_obs,
                "rewards": rewards,
                "dones": dones,
                "achieved_goal": achieved_goal,
                "next_achieved_goal": next_achieved_goal,
                "desired_goal": obs_dict["desired_goal"]
            }
    replay_buffer.add_records(record)


[[ 0.5380715  -0.86990213  0.46769086 -0.91408646]
 [ 0.76896137  0.4276612   0.4083679   0.23015888]
 [ 0.89100766  0.61377484  0.46597376 -0.43470734]
 [ 0.3697889   0.16326703  0.93437463  0.7651874 ]
 [-0.83715254  0.34797984  0.7814982  -0.35943627]
 [ 0.56747276 -0.1593227  -0.84747493 -0.4378694 ]
 [-0.06466938 -0.86335814  0.22814383 -0.88818085]
 [ 0.18131043  0.2523995  -0.66311294  0.6091487 ]
 [-0.34383842 -0.4155667  -0.08353846 -0.37420994]
 [ 0.09749154  0.8921424  -0.80625796 -0.13840319]
 [ 0.70780116 -0.7944362   0.13097104  0.5824863 ]
 [ 0.20735201 -0.05318557 -0.80771    -0.20892473]
 [-0.28231254  0.07447082 -0.6354129   0.10961482]
 [-0.57549524 -0.69257814 -0.34429732  0.46728572]
 [ 0.7347246   0.63446534 -0.26119724 -0.95630026]
 [ 0.545681    0.3569201  -0.39350155 -0.46391582]
 [-0.9028263   0.6733521  -0.4734923   0.23227873]
 [-0.90221435  0.913387    0.12695503  0.86038214]
 [ 0.5407396  -0.35031882  0.2265379   0.5070726 ]
 [-0.49449137 -0.72043645 -0.50

In [27]:
batch = replay_buffer.sample_batch(key_batch, 16, 4, "dones", ["future"])

In [28]:
batch.keys()

dict_keys(['observations', 'achieved_goal', 'next_observations', 'next_achieved_goal', 'desired_goal', 'rewards', 'dones', 'observations_future', 'achieved_goal_future', 'next_observations_future', 'next_achieved_goal_future', 'desired_goal_future', 'rewards_future', 'dones_future'])

In [ ]:
her = HindsightExperienceReplay(
    replay_strategy="future",
    replay_k=4,          # future_p = 1 - 1/(1 + k)
    env=envs
)


In [29]:
train_batch = her.process_batch(batch)

In [30]:
train_batch.keys()

dict_keys(['observations', 'achieved_goal', 'next_observations', 'next_achieved_goal', 'desired_goal', 'rewards', 'dones', 'observations_future', 'achieved_goal_future', 'next_observations_future', 'next_achieved_goal_future', 'desired_goal_future', 'rewards_future', 'dones_future', 'reward', 'her_mask'])

In [31]:
mask = train_batch["her_mask"]

In [32]:
batch["observations"]

tensor([[ 1.3418e+00,  7.4910e-01,  5.3473e-01,  3.9006e-04,  1.1703e-05,
         -5.2614e-06, -8.4646e-08,  2.4623e-05,  6.6556e-06, -2.9418e-06],
        [ 1.3418e+00,  7.4910e-01,  5.3473e-01,  3.9006e-04,  1.1703e-05,
         -5.2614e-06, -8.4646e-08,  2.4623e-05,  6.6556e-06, -2.9418e-06],
        [ 1.3418e+00,  7.4910e-01,  5.3473e-01,  3.9006e-04,  1.1703e-05,
         -5.2614e-06, -8.4646e-08,  2.4623e-05,  6.6556e-06, -2.9418e-06],
        [ 1.3418e+00,  7.4910e-01,  5.3473e-01,  3.9006e-04,  1.1703e-05,
         -5.2614e-06, -8.4646e-08,  2.4623e-05,  6.6556e-06, -2.9418e-06],
        [ 1.3418e+00,  7.4910e-01,  5.3473e-01,  3.9006e-04,  1.1703e-05,
         -5.2614e-06, -8.4646e-08,  2.4623e-05,  6.6556e-06, -2.9418e-06],
        [ 1.3418e+00,  7.4910e-01,  5.3473e-01,  3.9006e-04,  1.1703e-05,
         -5.2614e-06, -8.4646e-08,  2.4623e-05,  6.6556e-06, -2.9418e-06],
        [ 1.3418e+00,  7.4910e-01,  5.3473e-01,  3.9006e-04,  1.1703e-05,
         -5.2614e-06, -8.4646e-0

In [33]:
batch["achieved_goal"][mask]

tensor([[1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347]])

In [34]:
train_batch["desired_goal"][mask]

tensor([[1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347],
        [1.3418, 0.7491, 0.5347]])

In [35]:
batch["desired_goal"][mask]

tensor([[1.2877, 0.8741, 0.4161],
        [1.4082, 0.6607, 0.4025],
        [1.4006, 0.8743, 0.6789],
        [1.2269, 0.8055, 0.6402],
        [1.3214, 0.6461, 0.4615],
        [1.3052, 0.7366, 0.6838],
        [1.4045, 0.6918, 0.5244],
        [1.4885, 0.7103, 0.6680],
        [1.4695, 0.7616, 0.5682],
        [1.2877, 0.8741, 0.4161],
        [1.4082, 0.6607, 0.4025],
        [1.4279, 0.6512, 0.4228],
        [1.4805, 0.7319, 0.5538]])

In [36]:
train_batch["rewards"][mask]

tensor([-1., -1., -1., -1., -1., -1., -1., -1.,  0., -1., -1., -1., -1.])